In [ ]:
import torch
import torch.nn as nn
from torch import optim
from torch.utils.tensorboard import SummaryWriter
import os, time, warnings
import numpy as np
import glob, math
import matplotlib.pyplot as plt
# Ensure inline plotting in Jupyter Notebook
%matplotlib inline
# 更改工作目录到项目根目录
os.chdir('/home/liangzida/workspace/iTransformer')
from data_provider.data_factory import data_provider
from experiments.exp_basic import Exp_Basic
from utils.tools import EarlyStopping, adjust_learning_rate, visual
from utils.metrics import metric
from tqdm import tqdm

class Args():
    def __init__(self):
        self.data = 'custom'
        self.root_path = './dataset/electricity/'
        self.data_path = 'electricity.csv'
        self.features = 'M'
        self.target = 'OT'
        self.freq = 'h'
        
        self.batch_size = 32
        self.num_workers = 0
        self.embed = 'timeF'
        
        self.label_len = 1
        self.seq_len = 48
        self.pred_len = 48
        
        self.d_model = 96
        self.enc_layers=3
        self.dec_layers=3
        self.dropout=0.5
        self.bidirectional=True
        self.lstm_num_layers=2
        self.lstm_hidden_size=8
        self.lstm_resnet=True
        self.kld_loss_weight=0.00025
args = Args()
data_set, data_loader = data_provider(args, flag='train')

In [2]:
class Encoder(nn.Module):
    def __init__(self, input_dim, d_model, layer_num=3, dropout=0.5, activation=nn.LeakyReLU()):
        super(Encoder, self).__init__()
        self.activation = activation
        self.cnn1 = nn.Conv1d(in_channels=1, out_channels=16, kernel_size=3, stride=1, padding=1)
        self.dropout1 = nn.Dropout(dropout)
        self.pool1 = nn.MaxPool1d(kernel_size=2, stride=2)

        self.cnn2 = nn.Conv1d(in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.dropout2 = nn.Dropout(dropout)
        self.pool2 = nn.MaxPool1d(kernel_size=2, stride=2)

        self.cnn3 = nn.Conv1d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1)
        self.dropout3 = nn.Dropout(dropout)
        self.pool3 = nn.MaxPool1d(kernel_size=2, stride=2)

        self.flatten = nn.Flatten()
        self.mean_linear = nn.Linear(384, d_model)
        self.log_var_linear = nn.Linear(384, d_model)
        
    def forward(self, input):
        input = input.unsqueeze(1)
        output = self.cnn1(input)
        output = self.activation(output)
        # output = self.dropout1(output)
        output = self.pool1(output)

        output = self.cnn2(output)
        output = self.activation(output)
        # output = self.dropout2(output)
        output = self.pool2(output)

        output = self.cnn3(output)
        output = self.activation(output)
        # output = self.dropout3(output)
        output = self.pool3(output)

        output = self.flatten(output)
        mean = self.mean_linear(output)
        log_var = self.log_var_linear(output)
        return mean, log_var

class Decoder(nn.Module):
    def __init__(self, d_model, output_dim, layer_num=3, dropout=0.5, activation=nn.Sigmoid(), bidirectional=True, lstm_num_layers=2, hidden_size=8, resnet=True):
        super(Decoder, self).__init__()
        self.resnet = resnet
        self.layer_num = layer_num
        self.activation = activation
        self.rnns = nn.ModuleList()
        self.rnns.append(nn.LSTM(input_size=1, hidden_size=hidden_size, num_layers=lstm_num_layers, batch_first=True, bidirectional=bidirectional, dropout=dropout))
        for _ in range(self.layer_num-1):
            self.rnns.append(nn.LSTM(input_size=hidden_size*(1+bidirectional), hidden_size=hidden_size, num_layers=lstm_num_layers, batch_first=True, bidirectional=bidirectional, dropout=dropout))
        self.flatten = nn.Flatten()
        self.projection = nn.Linear(d_model*hidden_size*(1+bidirectional), output_dim)
        
    def forward(self, input):
        output = self.rnns[0](input.unsqueeze(-1))[0]
        output = self.activation(output)
        
        for i in range(self.layer_num-1):
            inner_output = self.rnns[i+1](output)[0]
            inner_output = self.activation(inner_output)
            output = (inner_output + output) if self.resnet else inner_output
        
        output = self.flatten(output)
        output = self.projection(output)
        return output

In [3]:
class Norm(nn.Module):
    def __init__(self):
        super(Norm, self).__init__()
        self.mean = None
        self.std = None
        
    def forward(self, input, flag):
        if flag=='normalize':
            self.mean = torch.mean(input, dim=0, keepdim=True)
            self.std = (torch.std(input, dim=0, keepdim=True) + 1e-8)
            return (input - self.mean) / self.std
        else:
            return input * self.std + self.mean

class encoder_decoder_small_patch(nn.Module):
    def __init__(self, input_dim, d_model, output_dim, enc_layers=3, dec_layers=3, dropout=0.5, bidirectional=True, lstm_num_layers=2, lstm_hidden_size=8, lstm_resnet=True):
        super(encoder_decoder_small_patch, self).__init__()
        self.input_dim = input_dim
        self.d_model = d_model
        self.norm = Norm()
        self.encoder = Encoder(input_dim, d_model, layer_num=enc_layers, dropout=dropout, activation=nn.LeakyReLU())
        self.decoder = Decoder(d_model, \
                                output_dim, \
                                layer_num=dec_layers, \
                                dropout=dropout, \
                                activation=nn.Sigmoid(), \
                                bidirectional=bidirectional, \
                                lstm_num_layers=lstm_num_layers, \
                                hidden_size=lstm_hidden_size, \
                                resnet=lstm_resnet)
        
    def log_density_gaussian(self, sample, mu, logvar):
        '''计算vae的损失函数'''
        kld_loss = torch.mean(-0.5 * torch.sum(1 + logvar - mu ** 2 - logvar.exp(), dim = 1), dim = 0)
        return kld_loss
    
    def normal_sample(self, mean, logvar):
        '''采样'''
        dist = torch.distributions.Normal(0, 1)
        eps = dist.sample(mean.shape).cuda()
        z = mean + torch.exp(.5*logvar) * eps
        return z
    
    def forward(self, input):
        output = self.norm(input, 'normalize')
        mean, logvar = self.encoder(output)
        z = self.normal_sample(mean, logvar)
        loss_vae = self.log_density_gaussian(z, mean, logvar)
        output = self.decoder(z)
        output = self.norm(output, 'denormalize')
        return output, loss_vae, mean, torch.exp(.5*logvar), z
    

In [ ]:
'''训练'''
device = 'cuda'
enc_dec_small_patch = encoder_decoder_small_patch(input_dim=args.seq_len, d_model=args.d_model, output_dim=args.seq_len, \
                                                    enc_layers=args.enc_layers, dec_layers=args.dec_layers, dropout=args.dropout,\
                                                    bidirectional=args.bidirectional, lstm_num_layers=args.lstm_num_layers, lstm_hidden_size=args.lstm_hidden_size,\
                                                    lstm_resnet=args.lstm_resnet).to(device)
enc_dec_small_patch = nn.DataParallel(enc_dec_small_patch, device_ids=[0, 1])

enc_dec_small_patch.train()
criterion_mse = nn.MSELoss()
# HuberLoss
criterion = nn.SmoothL1Loss()
optimizer = torch.optim.Adam(enc_dec_small_patch.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

dir_path = '/home/liangzida/workspace/iTransformer/junk/encoder/' + time.strftime("%Y-%m-%d-%H-%M-%S", time.localtime()) + '/'\
                + 'seqlen' + str(args.seq_len) + 'd_model' + str(args.d_model) + 'enc_layers' + str(args.enc_layers)\
                + 'dec_layers' + str(args.dec_layers) + 'lstm_hidden_size' + str(args.lstm_hidden_size)
os.makedirs(dir_path)
# 保存args
with open(dir_path + '/args.txt', 'w') as f:
    f.write(str(args.__dict__))
writer = SummaryWriter(dir_path)
for epoch_count in tqdm(range(250)):
    for ii, (x, y, x_mark, y_mark) in (enumerate(data_loader)):
        B, L, C = x.shape
        x = torch.reshape(x.to(device), shape=[B*C, L]).float()
        # y = torch.reshape(y.to(device)[:, -args.pred_len:, :], shape=[B*C, L]).float()
        optimizer.zero_grad()
        x_recover, loss_vae, mean, std, z = enc_dec_small_patch(x)
        loss_vae = loss_vae * args.kld_loss_weight
        smoothL1loss = criterion(x_recover, x)
        mse = criterion_mse(x_recover, x)
        loss = smoothL1loss + loss_vae.mean()
        writer.add_scalar('Loss/loss',         loss.mean().item(),       epoch_count * len(data_loader) + ii)
        writer.add_scalar('Loss/SmoothL1Loss', smoothL1loss.item(),      epoch_count * len(data_loader) + ii)
        writer.add_scalar('Loss/VAEloss',      loss_vae.mean().item(),   epoch_count * len(data_loader) + ii)
        writer.add_scalar('Loss/mse',          mse.item(),               epoch_count * len(data_loader) + ii)
        writer.add_scalar('Loss/mean',         mean.abs().mean().item(), epoch_count * len(data_loader) + ii)
        writer.add_scalar('Loss/log_std',      std.mean().item(),        epoch_count * len(data_loader) + ii)
        loss.backward()
        optimizer.step()
        scheduler.step()
writer.close()

In [ ]:
'''画出结果'''
enc_dec_small_patch.eval()
# Ensure inline plotting in Jupyter Notebook
%matplotlib inline
args = Args()
args.seq_len = 48
args.pred_len = 48
args.d_model = 96
data_set, data_loader = data_provider(args, flag='val')

for ii, (x, y, x_mark, y_mark) in enumerate(data_loader):
    B, L, C = x.shape
    x = torch.reshape(x, shape=[B*C, L]).to(device).float()
    y = torch.reshape(y[:, -args.pred_len:, :], shape=[B*C, L]).to(device).float()
    x_recover, _, _, _, _ = enc_dec_small_patch(x)
    # criterion = nn.MSELoss()
    # loss = criterion(x_recover, x)
    x = x.cpu().detach().numpy()
    x_recover = x_recover.cpu().detach().numpy()
    # 在同一张图上，按照列添加子图，画出前五个原始数据和重构数据
    fig, axs = plt.subplots(5, 1, figsize=(20, 10))
    for i in range(5):
        axs[i].plot(x[i], label='Original')
        axs[i].plot(x_recover[i], label='Reconstructed')
        axs[i].legend()
    plt.tight_layout()
    plt.show()
    break

In [37]:
'''保存'''
# 保存模型
dir_path = r'/home/liangzida/workspace/iTransformer/junk/forcast/2024-10-27-14-18-11/seqlen48d_model96coder3/'
torch.save(enc_dec_small_patch, dir_path + '/sucess_vae_model_huberloss.pth')
# 保存args
with open(dir_path + '/args.txt', 'w') as f:
    f.write(str(args.__dict__))

In [ ]:
'''加载模型'''
args = Args()
args.seq_len = 48
args.pred_len = 48
args.d_model = 96
coder_num = 3

data_set, data_loader = data_provider(args, flag='train')

device = 'cuda'
model_path = '/home/liangzida/workspace/iTransformer/junk/forcast/2024-10-26-19-43-36/seqlen48d_model96coder3/model.pth'
enc_dec_small_patch = torch.load(model_path)